# AC-CVaR 条件风险价值组合优化模型

**参考论文**: Liang Ju (2022), "The Optimization of the Portfolio Selection Based on AC-CVaR Model"  
**核心思想**: 以CVaR (Conditional Value-at-Risk / Expected Shortfall) 替代方差作为风险度量，更好捕捉尾部风险  
**优化目标**: 在给定收益约束下最小化CVaR，与均值-方差模型进行对比

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 策略原理

### VaR与CVaR

给定置信水平 $\alpha$ (如95%)，组合损失 $L = -R_p$：

**VaR (Value at Risk)**:
$$\text{VaR}_\alpha = \inf\{l : P(L \leq l) \geq \alpha\}$$

**CVaR (Conditional Value at Risk / Expected Shortfall)**:
$$\text{CVaR}_\alpha = E[L \mid L \geq \text{VaR}_\alpha] = \frac{1}{1-\alpha} \int_\alpha^1 \text{VaR}_u \, du$$

### AC-CVaR 优化模型

基于Rockafellar & Uryasev (2000)的线性化方法：

$$\min_{w, \zeta} \quad \zeta + \frac{1}{T(1-\alpha)} \sum_{t=1}^{T} \max(- w^T r_t - \zeta, \; 0)$$

$$\text{s.t.} \quad w^T \bar{r} \geq r_{\text{target}}, \quad \sum w_i = 1, \quad w_i \geq 0$$

其中 $\zeta$ 是辅助变量（近似VaR），$r_t$ 是第 $t$ 日的收益向量。

### 与均值-方差的区别

| 特性 | Mean-Variance | Mean-CVaR |
|------|:---:|:---:|
| 风险度量 | $\sigma^2$ (对称) | CVaR (仅尾部) |
| 分布假设 | 正态 | 任意 |
| 尾部风险 | 低估 | 准确捕捉 |
| 一致性风险度量 | 否 | 是 |

In [ ]:
# Cell 3: 动画 - 收益分布中的VaR和CVaR区域
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

def animate_var_cvar():
    """动画展示不同置信水平下VaR和CVaR区域的变化"""
    # 模拟非正态组合收益 (混合正态 -> 厚尾)
    n = 5000
    returns = np.concatenate([
        np.random.normal(0.0005, 0.015, int(n * 0.85)),
        np.random.normal(-0.005, 0.03, int(n * 0.15)),
    ])
    np.random.shuffle(returns)

    alphas = np.arange(0.80, 0.996, 0.01)
    frames = []

    for alpha in alphas:
        var_val = np.percentile(returns, (1 - alpha) * 100)
        tail = returns[returns <= var_val]
        cvar_val = tail.mean() if len(tail) > 0 else var_val

        # 直方图
        hist_vals, bin_edges = np.histogram(returns, bins=100, density=True)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        # 全部分布
        colors = ['rgba(33,150,243,0.6)' if b > var_val else 'rgba(244,67,54,0.7)'
                  for b in bin_centers]
        # CVaR区域更深
        colors = ['rgba(183,28,28,0.9)' if b <= cvar_val
                  else ('rgba(244,67,54,0.5)' if b <= var_val else 'rgba(33,150,243,0.5)')
                  for b in bin_centers]

        frames.append(go.Frame(
            data=[
                go.Bar(x=bin_centers, y=hist_vals, marker_color=colors, name='收益分布',
                       showlegend=False),
                go.Scatter(x=[var_val, var_val], y=[0, max(hist_vals)*1.1],
                           mode='lines', name=f'VaR({alpha:.0%})={var_val:.4f}',
                           line=dict(color='red', width=3, dash='dash')),
                go.Scatter(x=[cvar_val, cvar_val], y=[0, max(hist_vals)*1.1],
                           mode='lines', name=f'CVaR({alpha:.0%})={cvar_val:.4f}',
                           line=dict(color='darkred', width=3)),
            ],
            name=f'\u03b1={alpha:.0%}'
        ))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title='VaR & CVaR: 收益分布尾部风险度量',
        xaxis_title='日收益率', yaxis_title='概率密度',
        height=500, width=850,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 200}, 'transition': {'duration': 100}}]),
        ])],
        sliders=[dict(steps=[
            dict(args=[[f.name]], label=f.name, method='animate') for f in frames
        ])],
        annotations=[
            dict(text='\u2588 CVaR尾部 (深红)  \u2588 VaR~CVaR (红)  \u2588 正常 (蓝)',
                 xref='paper', yref='paper', x=0.5, y=1.08, showarrow=False, font=dict(size=11)),
        ],
    )
    return fig

fig = animate_var_cvar()
fig.show()

In [ ]:
# Cell 4: 导入依赖
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import minimize, linprog

from shared.data_fetcher import get_etf_daily, get_stock_daily, get_index_daily
from shared.constants import (SECTOR_ETFS, DEFAULT_START, DEFAULT_END,
                               RISK_FREE_RATE, TRADING_DAYS_PER_YEAR, INITIAL_CAPITAL)
from shared.metrics import summary_table
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)

np.random.seed(42)
print('All imports loaded successfully.')

In [ ]:
# Cell 5: 下载行业ETF数据
etf_names = list(SECTOR_ETFS.keys())
etf_codes = list(SECTOR_ETFS.values())

fallback_stocks = {
    '银行': '601398', '券商': '600030', '医药': '600276', '消费': '000858',
    '科技': '002415', '新能源': '300750', '军工': '600893', '地产': '001979',
}

close_dict = {}
for name, code in SECTOR_ETFS.items():
    df = get_etf_daily(code, DEFAULT_START, DEFAULT_END)
    if df is not None and len(df) > 100 and 'close' in df.columns:
        close_dict[name] = df['close']
    else:
        print(f'ETF {name}({code}) 失败, 使用备用个股 {fallback_stocks[name]}')
        df = get_stock_daily(fallback_stocks[name], DEFAULT_START, DEFAULT_END)
        if df is not None and len(df) > 100 and 'close' in df.columns:
            close_dict[name] = df['close']

prices_df = pd.DataFrame(close_dict)
prices_df.sort_index(inplace=True)
prices_df.dropna(how='all', inplace=True)
prices_df.ffill(inplace=True)
prices_df.dropna(inplace=True)

bench_df = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
bench_close = bench_df['close'] if len(bench_df) > 0 else None

print(f'数据区间: {prices_df.index[0].date()} ~ {prices_df.index[-1].date()}')
print(f'资产数量: {prices_df.shape[1]}, 交易日: {prices_df.shape[0]}')

In [ ]:
# Cell 6: CVaR优化 + Mean-Variance优化函数

def minimize_cvar(returns_matrix, alpha=0.95, target_return=None):
    """
    最小化CVaR (历史模拟法)
    基于Rockafellar & Uryasev线性化方法
    
    returns_matrix: (T, N) 每日收益率矩阵
    alpha: 置信水平
    target_return: 目标收益率约束 (日频)
    """
    T, N = returns_matrix.shape
    mean_ret = returns_matrix.mean(axis=0)

    # 决策变量: [w_1,...,w_N, zeta, u_1,...,u_T]
    # 其中 u_t >= -w^T r_t - zeta, u_t >= 0
    n_vars = N + 1 + T

    # 目标: min zeta + (1/(T*(1-alpha))) * sum(u_t)
    c = np.zeros(n_vars)
    c[N] = 1.0  # zeta 系数
    c[N+1:] = 1.0 / (T * (1 - alpha))  # u_t 系数

    # 不等式约束: -w^T r_t - zeta <= u_t  =>  -r_t^T w - zeta - u_t <= 0
    A_ub = np.zeros((T, n_vars))
    for t in range(T):
        A_ub[t, :N] = -returns_matrix[t, :]   # -r_t^T w
        A_ub[t, N] = -1.0                      # -zeta
        A_ub[t, N + 1 + t] = -1.0              # -u_t
    b_ub = np.zeros(T)

    # 等式约束: sum(w) = 1
    A_eq = np.zeros((1, n_vars))
    A_eq[0, :N] = 1.0
    b_eq = np.array([1.0])

    # 收益约束
    if target_return is not None:
        A_ret = np.zeros((1, n_vars))
        A_ret[0, :N] = -mean_ret
        A_ub = np.vstack([A_ub, A_ret])
        b_ub = np.append(b_ub, -target_return)

    # 边界: w >= 0, zeta无界, u_t >= 0
    bounds = [(0, 0.4)] * N + [(None, None)] + [(0, None)] * T

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                     bounds=bounds, method='highs')

    if result.success:
        return result.x[:N], result.x[N]  # weights, VaR
    else:
        return np.ones(N) / N, 0.0


def mean_variance_optimize(returns_matrix, target_return=None):
    """标准均值-方差优化 (最小方差)"""
    N = returns_matrix.shape[1]
    mu = returns_matrix.mean(axis=0)
    cov = np.cov(returns_matrix.T)

    def portfolio_var(w):
        return w @ cov @ w

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    if target_return is not None:
        constraints.append({'type': 'ineq', 'fun': lambda w: w @ mu - target_return})

    bounds = [(0, 0.4)] * N
    w0 = np.ones(N) / N
    result = minimize(portfolio_var, w0, method='SLSQP',
                      bounds=bounds, constraints=constraints)
    return result.x if result.success else w0


print('CVaR / MV 优化函数定义完成')

In [ ]:
# Cell 7: 滚动窗口回测 - CVaR vs MV

returns_df = prices_df.pct_change().dropna()
asset_names = list(prices_df.columns)
n_assets = len(asset_names)

lookback = 120
rebalance_freq = 20
alpha = 0.95

dates = returns_df.index

# CVaR策略
port_ret_cvar = pd.Series(0.0, index=dates)
weights_cvar = np.ones(n_assets) / n_assets
cvar_weight_hist = []

# MV策略
port_ret_mv = pd.Series(0.0, index=dates)
weights_mv = np.ones(n_assets) / n_assets

# 等权基准
port_ret_ew = pd.Series(0.0, index=dates)
weights_ew = np.ones(n_assets) / n_assets

rebal_dates = []

for i in range(lookback, len(dates)):
    daily_ret = returns_df.iloc[i].values

    port_ret_cvar.iloc[i] = weights_cvar @ daily_ret
    port_ret_mv.iloc[i] = weights_mv @ daily_ret
    port_ret_ew.iloc[i] = weights_ew @ daily_ret

    if (i - lookback) % rebalance_freq == 0:
        window_ret = returns_df.iloc[i-lookback:i].values
        target_ret = window_ret.mean(axis=0).mean()  # 平均日收益为目标

        # CVaR优化
        try:
            w_cvar, _ = minimize_cvar(window_ret, alpha=alpha, target_return=target_ret)
            weights_cvar = np.clip(w_cvar, 0, None)
            weights_cvar /= weights_cvar.sum() + 1e-10
        except Exception:
            weights_cvar = np.ones(n_assets) / n_assets

        # MV优化
        try:
            weights_mv = mean_variance_optimize(window_ret, target_return=target_ret)
            weights_mv = np.clip(weights_mv, 0, None)
            weights_mv /= weights_mv.sum() + 1e-10
        except Exception:
            weights_mv = np.ones(n_assets) / n_assets

        # 交易成本
        if len(rebal_dates) > 0:
            port_ret_cvar.iloc[i] -= 0.002
            port_ret_mv.iloc[i] -= 0.002

        rebal_dates.append(dates[i])
        cvar_weight_hist.append(weights_cvar.copy())

# 截取
port_ret_cvar = port_ret_cvar.iloc[lookback:]
port_ret_mv = port_ret_mv.iloc[lookback:]
port_ret_ew = port_ret_ew.iloc[lookback:]

equity_cvar = INITIAL_CAPITAL * (1 + port_ret_cvar).cumprod()
equity_mv = INITIAL_CAPITAL * (1 + port_ret_mv).cumprod()
equity_ew = INITIAL_CAPITAL * (1 + port_ret_ew).cumprod()

print(f'调仓次数: {len(rebal_dates)}')
print(f'CVaR终端净值: {equity_cvar.iloc[-1]:,.0f}')
print(f'MV终端净值:   {equity_mv.iloc[-1]:,.0f}')
print(f'等权终端净值:  {equity_ew.iloc[-1]:,.0f}')

In [ ]:
# Cell 8: 回测统计对比

bench_returns = None
if bench_close is not None:
    bench_close_aligned = bench_close.reindex(port_ret_cvar.index).ffill()
    bench_returns = bench_close_aligned.pct_change().fillna(0)

metrics_cvar = summary_table(port_ret_cvar, equity_cvar, bench_returns)
metrics_mv = summary_table(port_ret_mv, equity_mv, bench_returns)
metrics_ew = summary_table(port_ret_ew, equity_ew, bench_returns)

# 对比表格
compare_keys = ['年化收益率', '年化波动率', '夏普比率', '最大回撤', 'Calmar比率']
print(f'{"指标":<12} {"CVaR优化":<14} {"均值-方差":<14} {"等权基准":<14}')
print('-' * 56)
for k in compare_keys:
    print(f'{k:<12} {metrics_cvar.get(k, "N/A"):<14} {metrics_mv.get(k, "N/A"):<14} {metrics_ew.get(k, "N/A"):<14}')

# 尾部风险对比
print('\n=== 尾部风险对比 ===')
for name, rets in [('CVaR优化', port_ret_cvar), ('均值-方差', port_ret_mv), ('等权', port_ret_ew)]:
    var_95 = np.percentile(rets, 5)
    cvar_95 = rets[rets <= var_95].mean()
    print(f'{name}: VaR(95%)={var_95:.4f}, CVaR(95%)={cvar_95:.4f}')

In [ ]:
# Cell 9: 可视化
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 1) 三策略对比
plot_cumulative_comparison(
    {'CVaR优化': port_ret_cvar, '均值-方差': port_ret_mv, '等权基准': port_ret_ew},
    title='AC-CVaR vs Mean-Variance vs Equal-Weight'
)

# 2) 回撤对比
plot_drawdown(equity_cvar, title='CVaR策略 - 回撤')

# 3) 权重演变
import matplotlib.pyplot as plt
weight_df = pd.DataFrame(cvar_weight_hist, columns=asset_names,
                          index=rebal_dates[:len(cvar_weight_hist)])
fig, ax = plt.subplots(figsize=(14, 5))
weight_df.plot.area(ax=ax, alpha=0.75, colormap='Set2')
ax.set_title('CVaR策略 - 资产权重演变', fontsize=14, fontweight='bold')
ax.set_ylabel('权重')
ax.set_ylim(0, 1)
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9)
plt.tight_layout()
plt.show()

# 4) 绩效表
plot_metrics_table(metrics_cvar, title='AC-CVaR 策略绩效指标')

## 结果分析

### CVaR vs Mean-Variance 对比
1. **CVaR优化**在极端下跌时表现更好，因为它显式优化了尾部风险
2. **Mean-Variance**假设正态分布，低估了厚尾事件的损失
3. CVaR策略通常会更倾向于分散化配置，避免集中在波动率低但左尾风险大的资产上

### 实际应用注意
- CVaR优化的计算量较大（线性规划维度随样本数增长）
- $\alpha$ 置信水平的选择影响策略的保守程度
- 历史模拟法假设未来分布与历史相似，在市场结构突变时需警惕
- 可引入鲁棒CVaR (Worst-Case CVaR) 进一步增强策略稳定性